# Gold helper — SCD 1 / SCD 2 by checksum (LongChau poc method)

Same method as the poc repo (`common/hash_utils.py`, `common/scd_type_1.py`, `common/scd_type_2_new.py`), rewritten for Project 1:

| Column | Built from | Used for |
|---|---|---|
| `primary_checksum` | primary key columns (columns ending in `id` are upper-cased, as in poc) | MERGE key |
| `record_checksum` | every column except metadata, checksums and `exclude` | did anything change? |
| `scd_checksum` | the SCD 2 columns (`*` = same set as `record_checksum`) | major (new version) or minor (update in place) |

Metadata, never hashed: `etl_date`, `etl_by`, `current_flag`, `start_date`, `end_date`.

**SCD 1** `scd_type_1(df, table, primary_key)`: MERGE on `primary_checksum`, update only when `record_checksum` differs, insert new keys.

**SCD 2** `scd_type_2(df, table)` (poc "new helper"): rows whose `record_checksum` is not among the current (`current_flag = 'Y'`) rows are split into
- **minor**: same key, same `scd_checksum` → update the current row in place;
- **major**: same key, different `scd_checksum` → close the current row (`current_flag = 'N'`, `end_date = modified_at`) and insert a new one (`start_date = modified_at`);
- **new key** → insert with `start_date = created_at`.
One MERGE for minor updates, one MERGE (staged union with `merge_key`) for close + insert.

Differences from poc, on purpose:
1. The source must hold **one row per key** (checked, fails otherwise); poc only dedups in 2 notebooks.
2. A row whose `modified_at` is not newer than the current row's is skipped (SCD 1 and SCD 2), so a backfill of an old day can't overwrite newer values and can't create `start_date > end_date` (poc issue on `dim_family_member` / `dim_family_package_group`).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, ArrayType
from delta.tables import DeltaTable

METADATA_COLUMNS = ["etl_date", "etl_by", "current_flag", "start_date", "end_date"]
CHECKSUM_COLUMNS = ["primary_checksum", "record_checksum", "scd_checksum"]
MAX_DATE = "9999-12-31 23:59:59"


def _prepared(df, columns, upper_columns=()):
    """NULL -> '', struct/array -> JSON, everything else -> string; optional upper()."""
    out = []
    for c in columns:
        dtype = df.schema[c].dataType
        e = F.to_json(F.col(c)) if isinstance(dtype, (StructType, ArrayType)) else F.col(c).cast("string")
        e = F.coalesce(e, F.lit(""))
        out.append(F.upper(e) if c in upper_columns else e)
    return out


def add_checksums(df, primary_key, exclude=()):
    """primary_checksum (key) + record_checksum (all non-metadata columns minus `exclude`)."""
    skip = set(METADATA_COLUMNS + CHECKSUM_COLUMNS) | set(exclude)
    upper_cols = [c for c in primary_key if c.lower().endswith("id")]
    record_cols = [c for c in df.columns if c not in skip]
    return (df.withColumn("primary_checksum", F.md5(F.concat_ws("||", *_prepared(df, primary_key, upper_cols))))
              .withColumn("record_checksum", F.md5(F.concat_ws("||", *_prepared(df, record_cols, upper_cols)))))


def add_scd_checksum(df, scd_cols, exclude=()):
    """scd_checksum over the SCD 2 columns; '*' = every non-metadata column minus `exclude`."""
    skip = set(METADATA_COLUMNS + CHECKSUM_COLUMNS) | set(exclude)
    cols = [c for c in df.columns if c not in skip] if "*" in scd_cols else list(scd_cols)
    return df.withColumn("scd_checksum", F.md5(F.concat_ws("||", *_prepared(df, cols))))


def assert_one_row_per_key(df, name):
    dup = df.groupBy("primary_checksum").count().where("count > 1").count()
    if dup:
        raise ValueError(f"{name}: {dup} keys appear more than once in the source; dedup before the MERGE")

In [ ]:
def scd_type_1(source_df, table, primary_key, keep=("id",), order_col="modified_at"):
    """SCD 1: MERGE on primary_checksum, update only when record_checksum differs.
    `keep`: target columns never written by the MERGE (e.g. an IDENTITY id).
    `order_col`: only update when the source row is not older (>=), so a backfill of an old day can't overwrite newer
    values; >= still lets the same read update when a looked-up attribute changed (not in poc; None = poc behaviour)."""
    assert_one_row_per_key(source_df, table)
    cols = [c for c in source_df.columns if c not in keep]
    cond = "t.record_checksum <> s.record_checksum"
    if order_col:
        cond += f" AND (t.{order_col} IS NULL OR s.{order_col} >= t.{order_col})"
    (DeltaTable.forName(spark, table).alias("t")
        .merge(source_df.alias("s"), "t.primary_checksum = s.primary_checksum")
        .whenMatchedUpdate(condition=cond,
                           set={c: f"s.{c}" for c in cols if c not in primary_key})
        .whenNotMatchedInsert(values={c: f"s.{c}" for c in cols})
        .execute())

In [ ]:
def scd_type_2(source_df, table, effective_start_col="created_at", effective_end_col="modified_at", keep=("id",)):
    """SCD 2 (poc new-helper method). Returns a dict with the number of minor / major / new rows."""
    assert_one_row_per_key(source_df, table)
    target = DeltaTable.forName(spark, table)
    cols = [c for c in source_df.columns if c not in ("current_flag", "start_date", "end_date") and c not in keep]

    cur = (target.toDF().where("current_flag = 'Y'")
           .select("primary_checksum", "record_checksum",
                   F.col("scd_checksum").alias("_t_scd"), F.col(effective_end_col).alias("_t_mod")))

    # rows whose content is not already the current version
    changed = source_df.join(cur.select("record_checksum"), "record_checksum", "left_anti")
    j = changed.join(cur.drop("record_checksum"), "primary_checksum", "left")
    # guard: skip reads that are not newer than the current version (late / re-sent rows)
    j = j.where(F.col("_t_mod").isNull() | (F.col(effective_end_col) > F.col("_t_mod")))

    minor = j.where("_t_scd IS NOT NULL AND _t_scd = scd_checksum").select(*cols)
    major = j.where("_t_scd IS NOT NULL AND _t_scd <> scd_checksum").select(*cols)
    new = j.where("_t_scd IS NULL").select(*cols)
    counts = {"minor": minor.count(), "major": major.count(), "new": new.count()}

    # 1) minor: update the current row in place
    (target.alias("t")
        .merge(minor.alias("s"), "t.primary_checksum = s.primary_checksum AND t.current_flag = 'Y'")
        .whenMatchedUpdate(set={c: f"s.{c}" for c in cols})
        .execute())

    # 2) major + new: close the old version and insert the new one in ONE merge
    #    merge_key = key  -> matches the current row -> close it
    #    merge_key = NULL -> never matches            -> insert the new version
    staged = (
        new.withColumn("merge_key", F.col("primary_checksum"))
           .withColumn("start_date", F.col(effective_start_col).cast("timestamp"))
        .unionByName(major.withColumn("merge_key", F.lit(None).cast("string"))
                          .withColumn("start_date", F.col(effective_end_col).cast("timestamp")))
        .unionByName(major.withColumn("merge_key", F.col("primary_checksum"))
                          .withColumn("start_date", F.lit(None).cast("timestamp")))
        .withColumn("end_date", F.lit(MAX_DATE).cast("timestamp"))
        .withColumn("current_flag", F.lit("Y"))
    )
    (target.alias("t")
        .merge(staged.alias("s"), "t.primary_checksum = s.merge_key AND t.current_flag = 'Y'")
        .whenMatchedUpdate(set={"current_flag": "'N'", "end_date": f"s.{effective_end_col}"})
        .whenNotMatchedInsert(values={c: f"s.{c}" for c in cols + ["start_date", "end_date", "current_flag"]})
        .execute())
    return counts

In [ ]:
def stg_window(stg_table, load_type, from_date, to_date, key="channel_id"):
    """Read Silver (grain key x snapshot_ts) and keep the LAST read per key.
    incremental: snapshot_date (Vietnam day) in [from_date, to_date); full: every read."""
    from pyspark.sql.window import Window
    df = spark.table(stg_table)
    if load_type != "full":
        df = df.where((F.col("snapshot_date") >= F.lit(from_date)) & (F.col("snapshot_date") < F.lit(to_date)))
    w = Window.partitionBy(key).orderBy(F.col("snapshot_ts").desc(), F.col("etl_date").desc_nulls_last())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")


def default_dates(from_date, to_date, tz="Asia/Ho_Chi_Minh"):
    """Empty widgets -> yesterday .. tomorrow (Vietnam): poc uses yesterday .. today, but the job runs 3 times a day,
    so today's reads are included too; a rerun over the same window writes nothing (checksum)."""
    from datetime import datetime, timedelta
    from zoneinfo import ZoneInfo
    if not from_date.strip() and not to_date.strip():
        today = datetime.now(ZoneInfo(tz)).date()
        return str(today - timedelta(days=1)), str(today + timedelta(days=1))
    return from_date.strip(), to_date.strip()

In [ ]:
TZ = "Asia/Ho_Chi_Minh"


def vn_date(col_name):
    """Vietnam day of a UTC timestamp column."""
    return F.to_date(F.from_utc_timestamp(F.col(col_name), TZ))


def in_window(day_col, from_date, to_date):
    """day_col (a DATE Column) in [from_date, to_date)."""
    return (day_col >= F.lit(from_date).cast("date")) & (day_col < F.lit(to_date).cast("date"))


def changed_keys(*key_frames):
    """Union of the keys that changed in any source (DE pattern from GORIC-2370: union the changed keys of every
    source table, then re-read the full rows). Each frame has one column with the same name."""
    out = key_frames[0]
    for k in key_frames[1:]:
        out = out.unionByName(k)
    return out.where(F.col(out.columns[0]).isNotNull()).distinct()


def load_scd1(src, table, ddl, comment, primary_key, load_type, unknown, order_col="modified_at"):
    """Create the table if missing, then full -> overwrite (one atomic commit, poc full_refresh) or
    incremental -> scd_type_1 MERGE. `unknown`: {column: SQL literal} of the Unknown row, the first column is its key."""
    comment = comment.replace("'", "")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} ({ddl}) USING DELTA COMMENT '{comment}'")
    cols = spark.table(table).columns
    src = src.select(*cols)
    n_src = src.count()
    if load_type == "full":
        src.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    else:
        scd_type_1(src, table, primary_key, keep=(), order_col=order_col)
    v = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").select("operation", "operationMetrics").first()
    key_col, key_val = next(iter(unknown.items()))
    spark.sql(f"""
        INSERT INTO {table} ({", ".join(unknown)}) SELECT {", ".join(unknown.values())}
        WHERE NOT EXISTS (SELECT 1 FROM {table} WHERE {key_col} = {key_val})
    """)
    pk = ", ".join(primary_key)
    r = spark.sql(f"SELECT COUNT(*) AS n, COUNT(*) - COUNT(DISTINCT {pk}) AS dup FROM {table} WHERE {primary_key[0]} IS NOT NULL").first()
    if r["dup"]:
        raise ValueError(f"{table}: {r['dup']} duplicate {pk}")
    print(f"✅ {table}: {load_type}, source {n_src} rows, table {r['n']} rows · last op {v['operation']} "
          f"{ {k: x for k, x in v['operationMetrics'].items() if k in ('numTargetRowsUpdated', 'numTargetRowsInserted', 'numOutputRows')} }")